In [4]:
import recordlinkage
import pandas as pd
import numpy as np
from recordlinkage.datasets import load_febrl4 # fictitious datase for testing, this consists of two sets of 5000 identical records
from recordlinkage.preprocessing import phonetic

In [5]:
dfa, dfb, true_links = load_febrl4(return_links=True) # however the records, whilst all duplicates, are not identical.

In [6]:
print('Dataset A')
dfa.sort_index().head()

Dataset A


,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id
rec_id,,,,,,,,,,
rec-0-org,rachael,dent,1,knox street,lakewood estate,byford,4129,vic,19280722,1683994
rec-1-org,isabella,everett,25,pike place,rowethorpe,marsden,2152,nsw,19110816,6653129
rec-10-org,lachlan,reid,5,carrington road,legacy vlge,yagoona,2464,nsw,19500531,3232033
rec-100-org,hayden,stapley,38,tindale street,villa 2,cromer heights,4125,vic,NaN,4620080
rec-1000-org,victoria,zbierski,70,wybalena grove,inverneath,paralowie,5065,nsw,19720503,1267612


In [7]:
print('Dataset B')
dfb.sort_index().head()

Dataset B


,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth,soc_sec_id
rec_id,,,,,,,,,,
rec-0-dup-0,rachael,dent,4,knox street,lakewood estate,byford,4129,vic,19280722,1683994
rec-1-dup-0,isabella,everett,25,pike mlace,rowethorpe,marsden,2152,nsw,19110816,6653129
rec-10-dup-0,lachlnn,reid,5,carrington road,legacy vlge,yagoona,2446,nsw,19500531,3232033
rec-100-dup-0,hayden,stapley,NaN,tindale street,villa 2,cromer heights,4125,vic,NaN,4620080
rec-1000-dup-0,victoria,zbierski,70,wybalena grove,inverbeath,paralowie,5065,nsw,19720503,1267612


In [8]:
print('dataset A')
print(dfa.isnull().sum())
print(30*'-')
print('dataset B')
print(dfb.isnull().sum())   

dataset A
given_name       112
surname           48
street_number    158
address_1         98
address_2        420
suburb            55
postcode           0
state             50
date_of_birth     94
soc_sec_id         0
dtype: int64
------------------------------
dataset B
given_name       234
surname          102
street_number    287
address_1        220
address_2        851
suburb           106
postcode           0
state            107
date_of_birth    199
soc_sec_id         0
dtype: int64


### Preprocess "Data cleaning"

In [9]:
# Example
print("\nNote how all four katies are soundex'd to the same value...")
katies = pd.Series(["Katy", "Kaytee", "Katie", "Katee"])
phonetic_katies = phonetic(katies, "soundex")

for i in range(4):
    print(katies[i] + " --> " + phonetic_katies[i])


Note how all four katies are soundex'd to the same value...
Katy --> K300
Kaytee --> K300
Katie --> K300
Katee --> K300


In [10]:
# Add a phonetic version of the first and last names to the two datasets
dfa['phonetic_surname'] = phonetic(dfa['surname'], 'soundex')
dfa['phonetic_given_name'] = phonetic(dfa['given_name'], 'soundex')
dfb['phonetic_surname'] = phonetic(dfb['surname'], 'soundex')
dfb['phonetic_given_name'] = phonetic(dfb['given_name'], 'soundex')

# Adding initials
dfa['initials'] = dfa['given_name'].str[0] + dfa['surname'].str[0]
dfb['initials'] = dfb['given_name'].str[0] + dfb['surname'].str[0]

In [11]:
# Casting social security number as "numeric"
dfa['soc_sec_id'] = pd.to_numeric(dfa['soc_sec_id'])
dfb['soc_sec_id'] = pd.to_numeric(dfb['soc_sec_id'])

In [12]:
print(dfa['soc_sec_id'].dtype)
print(dfb['soc_sec_id'].dtype)

int64
int64


In [13]:

dfa[['given_name', 'surname', 'soc_sec_id', 'phonetic_surname', 'phonetic_given_name', 'initials']].head(3)

,given_name,surname,soc_sec_id,phonetic_surname,phonetic_given_name,initials
rec_id,,,,,,
rec-1070-org,michaela,neumann,5304218,N550,M240,mn
rec-1016-org,courtney,painter,4066625,P536,C635,cp
rec-4405-org,charles,green,4365168,G650,C642,cg


In [14]:
dfb[['given_name', 'surname', 'soc_sec_id', 'phonetic_surname', 'phonetic_given_name', 'initials']].head(3)

,given_name,surname,soc_sec_id,phonetic_surname,phonetic_given_name,initials
rec_id,,,,,,
rec-561-dup-0,elton,NaN,1551941,NaN,E435,NaN
rec-2642-dup-0,mitchell,maxon,8859999,M250,M324,mm
rec-608-dup-0,NaN,white,9731855,W300,NaN,NaN


### Indexing/Blocking

In [15]:
print(dfa.shape)
print(dfb.shape)

(5000, 13)
(5000, 13)


In [16]:
indexer = recordlinkage.Index()
indexer.block('initials')
candidate_links = indexer.index(dfa, dfb)
candidate_links

MultiIndex([('rec-1070-org', 'rec-2820-dup-0'),
            ('rec-1070-org',  'rec-684-dup-0'),
            ('rec-1070-org', 'rec-2942-dup-0'),
            ('rec-1070-org', 'rec-2283-dup-0'),
            ('rec-1070-org',  'rec-992-dup-0'),
            ('rec-1070-org', 'rec-3535-dup-0'),
            ('rec-1070-org', 'rec-2231-dup-0'),
            ('rec-1070-org', 'rec-1889-dup-0'),
            ('rec-1070-org', 'rec-2033-dup-0'),
            ('rec-1070-org', 'rec-4515-dup-0'),
            ...
            (  'rec-66-org', 'rec-1221-dup-0'),
            (  'rec-66-org',   'rec-66-dup-0'),
            (  'rec-66-org', 'rec-2826-dup-0'),
            (  'rec-66-org', 'rec-2919-dup-0'),
            (  'rec-66-org', 'rec-3838-dup-0'),
            (  'rec-66-org', 'rec-2399-dup-0'),
            (  'rec-66-org', 'rec-1556-dup-0'),
            (  'rec-66-org', 'rec-1957-dup-0'),
            (  'rec-66-org', 'rec-1928-dup-0'),
            (  'rec-66-org', 'rec-3380-dup-0')],
           names=['rec_

In [17]:
# indexx = recordlinkage.Index()
# indexx.block(['initials', 'phonetic_given_name']) # Blocking on multiple variables will reduce the number of record pairs even further.
# indexx.index(dfa, dfb)

In [18]:
dfa.loc['rec-1070-org']

given_name                   michaela
surname                       neumann
street_number                       8
address_1              stanley street
address_2                       miami
suburb                  winston hills
postcode                         4223
state                             nsw
date_of_birth                19151111
soc_sec_id                    5304218
phonetic_surname                 N550
phonetic_given_name              M240
initials                           mn
Name: rec-1070-org, dtype: object

In [19]:
dfb.loc['rec-2942-dup-0']

given_name                    mikayla
surname                        nguyen
street_number                      11
address_1               jaunceyucourt
address_2              karinyag ardns
suburb                        helidon
postcode                         5197
state                             nsw
date_of_birth                19030122
soc_sec_id                    3964863
phonetic_surname                 N250
phonetic_given_name              M240
initials                           mn
Name: rec-2942-dup-0, dtype: object

### Comparing

In [20]:
dfa.soc_sec_id

rec_id
rec-1070-org    5304218
rec-1016-org    4066625
rec-4405-org    4365168
rec-1288-org    9239102
rec-3585-org    7207688
                 ...   
rec-2153-org    7676186
rec-1604-org    4971506
rec-1003-org    8927667
rec-4883-org    6039042
rec-66-org      6375537
Name: soc_sec_id, Length: 5000, dtype: int64

In [21]:
compare = recordlinkage.Compare()
compare.exact('phonetic_surname', 'phonetic_surname', label='phonetic_surname')
compare.exact('phonetic_given_name', 'phonetic_given_name', label='phonetic_given_name')
compare.string('given_name', 'given_name', method='jarowinkler', label='given_name')
compare.string('surname', 'surname', method='jarowinkler', label='surname')
compare.string('suburb', 'suburb', method='jarowinkler', label='suburb')
compare.string('state', 'state', method='jarowinkler', label='state') # try it with the levenshtein algorithm
compare.string('address_1', 'address_1', method='jarowinkler', label='address_1')
compare.numeric('soc_sec_id', 'soc_sec_id', method='linear', label='soc_sec_id') # default algorithm linear anyways I'll write it
posible_matches = compare.compute(candidate_links, dfa, dfb)
posible_matches

phonetic_surname  phonetic_given_name  \
rec_id_1     rec_id_2                                                
rec-1070-org rec-2820-dup-0                 0                    0   
             rec-684-dup-0                  0                    0   
             rec-2942-dup-0                 0                    1   
             rec-2283-dup-0                 0                    0   
             rec-992-dup-0                  0                    0   
...                                       ...                  ...   
rec-66-org   rec-2399-dup-0                 0                    1   
             rec-1556-dup-0                 0                    0   
             rec-1957-dup-0                 0                    0   
             rec-1928-dup-0                 0                    1   
             rec-3380-dup-0                 0                    0   

                             given_name   surname    suburb  state  address_1  \
rec_id_1     rec_id_2                                                           
rec-1070-org rec-2820-dup-0    0.638889  0.523810  0.410256    1.0   0.571429   
             rec-684-dup-0     0.638889  0.642857  0.485897    1.0   0.826840   
             rec-2942-dup-0    0.823810  0.642857  0.442002    1.0   0.574481   
             rec-2283-dup-0    0.648148  0.676190  0.529915    0.0   0.383117   
             rec-992-dup-0     0.583333  0.539683  0.646724    0.0   0.515873   
...                                 ...       ...       ...    ...        ...   
rec-66-org   rec-2399-dup-0    0.600000  0.618519  0.569795    1.0   0.687831   
             rec-1556-dup-0    0.455556  0.805291  0.516667    1.0   0.546958   
             rec-1957-dup-0    0.600000  0.611111  0.541667    0.0   0.575941   
             rec-1928-dup-0    0.633333  0.644444  0.444444    1.0   0.667507   
             rec-3380-dup-0    0.483333  0.407407  0.507576    0.0   0.646867   

                             soc_sec_id  
rec_id_1     rec_id_2                    
rec-1070-org rec-2820-dup-0         0.0  
             rec-684-dup-0          0.0  
             rec-2942-dup-0         0.0  
             rec-2283-dup-0         0.0  
             rec-992-dup-0          0.0  
...                                 ...  
rec-66-org   rec-2399-dup-0         0.0  
             rec-1556-dup-0         0.0  
             rec-1957-dup-0         0.0  
             rec-1928-dup-0         0.0  
             rec-3380-dup-0         0.0  

[103510 rows x 8 columns]

### Classification

In [22]:
# Try all scores 0 to 7 in order to see the effect on precision and reacl...

for i in range(8):
    matches = posible_matches[posible_matches.sum(axis=1) > i]
    precision = recordlinkage.precision(true_links, matches)
    recall = recordlinkage.recall(true_links, matches)
    print(f'When score is {i} precision is {precision} and recall is {recall}')

When score is 0 precision is 0.03605448748913148 and recall is 0.7464
When score is 1 precision is 0.03606737989620481 and recall is 0.7464
When score is 2 precision is 0.042211464507080486 and recall is 0.7464
When score is 3 precision is 0.11120381406436233 and recall is 0.7464
When score is 4 precision is 0.4544457978075518 and recall is 0.7462
When score is 5 precision is 0.9063871282301317 and recall is 0.7436
When score is 6 precision is 0.9858451290591174 and recall is 0.7104
When score is 7 precision is 1.0 and recall is 0.4876


### Using Support Vector Machine

In [23]:
true_links

MultiIndex([(   'rec-0-org',    'rec-0-dup-0'),
            (   'rec-1-org',    'rec-1-dup-0'),
            (   'rec-2-org',    'rec-2-dup-0'),
            (   'rec-3-org',    'rec-3-dup-0'),
            (   'rec-4-org',    'rec-4-dup-0'),
            (   'rec-5-org',    'rec-5-dup-0'),
            (   'rec-6-org',    'rec-6-dup-0'),
            (   'rec-7-org',    'rec-7-dup-0'),
            (   'rec-8-org',    'rec-8-dup-0'),
            (   'rec-9-org',    'rec-9-dup-0'),
            ...
            ('rec-4990-org', 'rec-4990-dup-0'),
            ('rec-4991-org', 'rec-4991-dup-0'),
            ('rec-4992-org', 'rec-4992-dup-0'),
            ('rec-4993-org', 'rec-4993-dup-0'),
            ('rec-4994-org', 'rec-4994-dup-0'),
            ('rec-4995-org', 'rec-4995-dup-0'),
            ('rec-4996-org', 'rec-4996-dup-0'),
            ('rec-4997-org', 'rec-4997-dup-0'),
            ('rec-4998-org', 'rec-4998-dup-0'),
            ('rec-4999-org', 'rec-4999-dup-0')],
           length=5000)

In [24]:
posible_matches

phonetic_surname  phonetic_given_name  \
rec_id_1     rec_id_2                                                
rec-1070-org rec-2820-dup-0                 0                    0   
             rec-684-dup-0                  0                    0   
             rec-2942-dup-0                 0                    1   
             rec-2283-dup-0                 0                    0   
             rec-992-dup-0                  0                    0   
...                                       ...                  ...   
rec-66-org   rec-2399-dup-0                 0                    1   
             rec-1556-dup-0                 0                    0   
             rec-1957-dup-0                 0                    0   
             rec-1928-dup-0                 0                    1   
             rec-3380-dup-0                 0                    0   

                             given_name   surname    suburb  state  address_1  \
rec_id_1     rec_id_2                                                           
rec-1070-org rec-2820-dup-0    0.638889  0.523810  0.410256    1.0   0.571429   
             rec-684-dup-0     0.638889  0.642857  0.485897    1.0   0.826840   
             rec-2942-dup-0    0.823810  0.642857  0.442002    1.0   0.574481   
             rec-2283-dup-0    0.648148  0.676190  0.529915    0.0   0.383117   
             rec-992-dup-0     0.583333  0.539683  0.646724    0.0   0.515873   
...                                 ...       ...       ...    ...        ...   
rec-66-org   rec-2399-dup-0    0.600000  0.618519  0.569795    1.0   0.687831   
             rec-1556-dup-0    0.455556  0.805291  0.516667    1.0   0.546958   
             rec-1957-dup-0    0.600000  0.611111  0.541667    0.0   0.575941   
             rec-1928-dup-0    0.633333  0.644444  0.444444    1.0   0.667507   
             rec-3380-dup-0    0.483333  0.407407  0.507576    0.0   0.646867   

                             soc_sec_id  
rec_id_1     rec_id_2                    
rec-1070-org rec-2820-dup-0         0.0  
             rec-684-dup-0          0.0  
             rec-2942-dup-0         0.0  
             rec-2283-dup-0         0.0  
             rec-992-dup-0          0.0  
...                                 ...  
rec-66-org   rec-2399-dup-0         0.0  
             rec-1556-dup-0         0.0  
             rec-1957-dup-0         0.0  
             rec-1928-dup-0         0.0  
             rec-3380-dup-0         0.0  

[103510 rows x 8 columns]

In [25]:
clasifier = recordlinkage.SVMClassifier()
clasifier.fit(posible_matches, true_links)
predictions = clasifier.predict(posible_matches)
predictions

MultiIndex([('rec-1016-org', 'rec-1016-dup-0'),
            ('rec-4405-org', 'rec-4405-dup-0'),
            ('rec-1288-org', 'rec-1288-dup-0'),
            ('rec-3585-org', 'rec-3585-dup-0'),
            ( 'rec-298-org',  'rec-298-dup-0'),
            ('rec-2404-org', 'rec-2404-dup-0'),
            ( 'rec-453-org',  'rec-453-dup-0'),
            ('rec-4866-org', 'rec-4866-dup-0'),
            ('rec-4302-org', 'rec-4302-dup-0'),
            ( 'rec-420-org',  'rec-420-dup-0'),
            ...
            ('rec-4521-org', 'rec-4521-dup-0'),
            ( 'rec-715-org',  'rec-715-dup-0'),
            ( 'rec-674-org',  'rec-674-dup-0'),
            ('rec-1916-org', 'rec-1916-dup-0'),
            ('rec-2792-org', 'rec-2792-dup-0'),
            ('rec-2613-org', 'rec-2613-dup-0'),
            ('rec-2235-org', 'rec-2235-dup-0'),
            ('rec-3877-org', 'rec-3877-dup-0'),
            ('rec-4883-org', 'rec-4883-dup-0'),
            (  'rec-66-org',   'rec-66-dup-0')],
           names=['rec_

In [34]:
print(f"Precision -> {recordlinkage.precision(true_links, predictions)}")
print(f"recall -> {recordlinkage.recall(true_links, predictions)}")

Precision -> 0.998652654271086
recall -> 0.7412


In [35]:
# clasifier2 = recordlinkage.LogisticRegressionClassifier()
# clasifier2.fit(posible_matches, true_links)
# predictions2 = clasifier2.predict(posible_matches)
# predictions2

In [36]:
# print(f"Precision -> {recordlinkage.precision(true_links, predictions2)}")
# print(f"Recall -> {recordlinkage.recall(true_links, predictions2)}")

In [37]:
# for x in predictions:
#     if(x not in predictions2):
#         print(x)

In [38]:
# false positive
print('Number of false positives: ' + str(recordlinkage.false_positives(true_links, predictions)))

Number of false positives: 5


In [43]:
false_positive = predictions.difference(true_links)
false_positive

MultiIndex([('rec-1153-org', 'rec-2499-dup-0'),
            ('rec-1546-org', 'rec-3587-dup-0'),
            ('rec-2499-org', 'rec-1153-dup-0'),
            ('rec-3587-org', 'rec-1546-dup-0'),
            ('rec-3959-org', 'rec-4295-dup-0')],
           )

In [48]:
dfa.loc['rec-1546-org']

given_name                  emiily
surname                      white
street_number                   12
address_1              crago place
address_2                  rocklea
suburb                      mosman
postcode                      3300
state                          vic
date_of_birth             19390429
soc_sec_id                 6209600
phonetic_surname              W300
phonetic_given_name           E540
initials                        ew
Name: rec-1546-org, dtype: object

In [49]:
dfb.loc['rec-3587-dup-0']

given_name                  emiily
surname                      white
street_number                    1
address_1              marou place
address_2                      NaN
suburb                     marsden
postcode                      3412
state                          vic
date_of_birth             19510901
soc_sec_id                 6646759
phonetic_surname              W300
phonetic_given_name           E540
initials                        ew
Name: rec-3587-dup-0, dtype: object

In [53]:
print("Number of false negatives: " + str(recordlinkage.false_negatives(true_links, predictions)))

Number of false negatives: 1294


In [54]:
false_negatives = true_links.difference(predictions)
false_negatives

MultiIndex([('rec-1003-org', 'rec-1003-dup-0'),
            ('rec-1007-org', 'rec-1007-dup-0'),
            ('rec-1009-org', 'rec-1009-dup-0'),
            ('rec-1010-org', 'rec-1010-dup-0'),
            ('rec-1011-org', 'rec-1011-dup-0'),
            ('rec-1017-org', 'rec-1017-dup-0'),
            ('rec-1040-org', 'rec-1040-dup-0'),
            ('rec-1046-org', 'rec-1046-dup-0'),
            ('rec-1053-org', 'rec-1053-dup-0'),
            ('rec-1054-org', 'rec-1054-dup-0'),
            ...
            ( 'rec-959-org',  'rec-959-dup-0'),
            ( 'rec-979-org',  'rec-979-dup-0'),
            (  'rec-98-org',   'rec-98-dup-0'),
            ( 'rec-981-org',  'rec-981-dup-0'),
            ( 'rec-982-org',  'rec-982-dup-0'),
            ( 'rec-983-org',  'rec-983-dup-0'),
            ( 'rec-987-org',  'rec-987-dup-0'),
            (  'rec-99-org',   'rec-99-dup-0'),
            ( 'rec-992-org',  'rec-992-dup-0'),
            ( 'rec-994-org',  'rec-994-dup-0')],
           length=1294)

In [55]:
dfa.loc['rec-1003-org']

given_name                  bradley
surname                    matthews
street_number                     2
address_1              jondol place
address_2              horseshoe ck
suburb                  jacobs well
postcode                       7018
state                            sa
date_of_birth              19481122
soc_sec_id                  8927667
phonetic_surname               M320
phonetic_given_name            B634
initials                         bm
Name: rec-1003-org, dtype: object

In [56]:
dfb.loc['rec-1003-dup-0']

given_name                 matthews
surname                     bradley
street_number                     2
address_1              jondol place
address_2              horsesheo ck
suburb                  jacobs qell
postcode                       7018
state                            sa
date_of_birth              19481122
soc_sec_id                  8927667
phonetic_surname               B634
phonetic_given_name            M320
initials                         mb
Name: rec-1003-dup-0, dtype: object